# MLflow: autologging vs manual tracking — side-by-side

This notebook compares two approaches to experiment tracking with MLflow: **manual tracking** (explicit `log_param`, `log_metric`, `log_artifact` calls) and **autologging** (automatic capture via `mlflow.autolog()`). Both approaches are run against the same model — a scikit-learn `RandomForestClassifier` on the Iris dataset — and results are inspected from the tracking store.

The goal is to see what each approach captures, where they differ, and when you'd pick one over the other.

## Setup

In [ ]:
import mlflow
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
mlflow.set_tracking_uri("sqlite:///mlflow_compare.db")

Load the Iris dataset — small enough to iterate quickly.

In [ ]:
data = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")

---
## Approach 1 — Manual tracking

This is the explicit approach: you control every `log_param`, `log_metric`, and `log_artifact` call. Nothing gets logged unless you write the call. This gives full control but requires more boilerplate.

In [ ]:
experiment_name = "autolog-vs-manual"
mlflow.set_experiment(experiment_name)

with mlflow.start_run(run_name="manual-tracking") as run_manual:
    manual_run_id = run_manual.info.run_id

    # --- params ---
    params = {
        "n_estimators": 100,
        "max_depth": 5,
        "min_samples_split": 2,
        "random_state": 42,
        "model_type": "RandomForestClassifier",
    }
    for k, v in params.items():
        mlflow.log_param(k, v)

    # --- train ---
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)

    # --- evaluate ---
    y_pred = model.predict(X_test)
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(y_test, y_pred, average="macro"),
        "recall_macro": recall_score(y_test, y_pred, average="macro"),
        "f1_macro": f1_score(y_test, y_pred, average="macro"),
    }
    mlflow.log_metrics(metrics)

    # --- artifact ---
    model_path = "manual_model.pkl"
    mlflow.sklearn.log_model(model, "model")

    print(f"Manual run {manual_run_id[:8]} complete")
    print(f"  Params logged: {len(params)}")
    print(f"  Metrics logged: {len(metrics)}")
    print(f"  Artifact: model/")

---
## Approach 2 — Autologging

With `mlflow.autolog()`, MLflow hooks into scikit-learn's `fit()` method automatically. Parameters, metrics, and the model artifact are captured without explicit logging calls. The trade-off: you get defaults unless you configure the autolog hook.

In [ ]:
mlflow.autolog(log_models=True, log_datasets=False, silent=True)

with mlflow.start_run(run_name="autolog-tracking") as run_auto:
    auto_run_id = run_auto.info.run_id

    model_auto = RandomForestClassifier(
        n_estimators=100, max_depth=5, min_samples_split=2, random_state=42
    )
    model_auto.fit(X_train, y_train)

    y_pred_auto = model_auto.predict(X_test)
    auto_acc = accuracy_score(y_test, y_pred_auto)

    print(f"Autolog run {auto_run_id[:8]} complete")
    print(f"  Test accuracy: {auto_acc:.4f}")

mlflow.autolog(disable=True)

---
## Comparison

Let's query the tracking store and see what each run actually captured.

In [ ]:
def describe_run(run_id, label):
    client = mlflow.tracking.MlflowClient()
    run = client.get_run(run_id)
    params = run.data.params
    metrics = run.data.metrics
    artifacts = client.list_artifacts(run_id)
    print(f"\n{'='*50}")
    print(f"  {label}  (run_id: {run_id[:8]})")
    print(f"{'='*50}")
    print(f"  Params ({len(params)}):")
    for k, v in sorted(params.items()):
        print(f"    {k}: {v}")
    print(f"  Metrics ({len(metrics)}):")
    for k, v in sorted(metrics.items()):
        print(f"    {k}: {v:.4f}")
    print(f"  Artifacts ({len(artifacts)}):")
    for a in artifacts:
        print(f"    {a.path}")

describe_run(manual_run_id, "MANUAL TRACKING")
describe_run(auto_run_id, "AUTOLOGGING")

### Key differences observed

| Aspect | Manual | Autolog |
|--------|--------|---------|
| **Param capture** | Only params I explicitly logged | Automatically captured all sklearn `RandomForestClassifier` constructor args plus training metadata (e.g. `random_state`, `n_estimators`, `max_depth`) |
| **Metrics** | I chose accuracy + precision/recall/f1 | Autolog logs training metrics (e.g. `training_score`) and sometimes evaluation metrics depending on the estimator |
| **Model artifact** | Logged explicitly via `log_model` | Logged automatically when `log_models=True` (default) |
| **Extra metadata** | None beyond what I wrote | Dataset digest, estimator class name, time-elapsed info — some of which is useful, some is noise for quick comparisons |
| **Code overhead** | ~15 lines of logging calls | Zero — one `mlflow.autolog()` call before training |

Autolog captures *more* params and metadata than manual tracking typically does. Whether that's helpful or cluttered depends on context — during iterative exploration the extra signal helps; in a polished report the manual approach lets you curate what appears.

## Verify — query runs via the MLflow Client

Check that both runs appear in the experiment and that metrics are queryable programmatically.

In [ ]:
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(experiment_name)

if experiment:
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["attributes.start_time desc"],
    )
    print(f"Experiment: {experiment.name} (id: {experiment.experiment_id})")
    print(f"Total runs: {len(runs)}")
    for r in runs:
        acc = r.data.metrics.get("accuracy") or r.data.metrics.get("training_score")
        print(f"  - {r.info.run_name}: accuracy={acc:.4f}" if acc else f"  - {r.info.run_name}")
else:
    print(f"Experiment '{experiment_name}' not found — runs may not have been created.")

## Summary

- **Autologging** is the quicker path — one call before training and everything is captured. It's a good default for the exploration phase, especially when iterating on models rapidly.
- **Manual tracking** takes more code but gives precise control over what appears in the tracking UI. It's useful when preparing reproducible baselines or when you only want a curated set of metrics logged.
- You can mix both within a project: autolog for broad capture during development, manual overrides for specific runs that need custom artifacts or fine-grained control.

One thing I'm not fully sure about: autolog's metric naming can vary between sklearn versions. The docs say `training_score` is standard, but I've seen it logged under different keys depending on the classifier type. This is something to verify when switching models.